In [3]:
%pip install voyageai rank_bm25 numpy

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached charset_normalizer-3.5.0-cp314-cp314-win_amd64.whl.metadata (44 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached huggingface_hub-1.27.0-py3-none-any.whl.metadata (16 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached tqdm-4.70.0-py3-none-any.whl.metadata (57 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
   --- ------------------------------------ 1.0/12.6 MB 3.5 MB/s eta 0:00:04
   --------- ------------------------------ 2.9/12.6 MB 5.7 MB/s eta 0:00:02
   -------------- ----------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from anthropic import Anthropic

client = Anthropic()

full_contract = """Nimbus Cloud Services. Contract NCS-2025-0142. Effective
Sept 25 2025. Provides primary cloud infrastructure hosting including
compute, storage, and managed database services. Auto-renews for
successive 12-month terms unless either party provides written notice
of cancellation at least 30 days before the end date. Annual value
$63,000."""

bare_chunk = "Auto-renews for successive 12-month terms unless either party provides written notice of cancellation at least 30 days before the end date."

response = client.messages.create(
    model="claude-sonnet-5",
    max_tokens=100,
    messages=[{
        "role": "user",
        "content": (
            f"Full document:\n{full_contract}\n\n"
            f"Chunk to contextualize:\n{bare_chunk}\n\n"
            "Write a one-sentence context (50-100 tokens) that situates "
            "this chunk within the document. Prepend it to the chunk. "
            "Output only the final contextualized chunk, nothing else."
        ),
    }],
)
print(next(b.text for b in response.content if b.type == "text"))

This clause appears in the agreement's term and termination section, establishing the conditions under which the contract automatically continues beyond its initial period: Auto-renews for successive 12-month terms unless either party provides written notice of cancellation at least 30 days before the end date.


In [5]:
import numpy as np
from voyageai import Client

voyage = Client()

contracts = {
    "Nimbus Cloud Services": "Provides primary cloud infrastructure hosting including compute, storage, and managed database services for production workloads.",
    "BrightPath Security Solutions": "Provides endpoint security tooling, vulnerability scanning, and quarterly compliance reporting.",
    "Vertex Networking Group": "Provides managed WAN circuits and telecom support across all regional offices.",
    "Alderwood Office Supplies": "Supplies IT peripherals, cabling, and consumables for the office and data closet.",
    "Fixed-Term Consulting LLC": "Provides contract engineering support for the infrastructure modernization project.",
}

names = list(contracts.keys())
docs = list(contracts.values())

doc_result = voyage.embed(docs, model="voyage-4", input_type="document")
doc_vectors = np.array(doc_result.embeddings)

query = "Which vendor helps keep our systems safe from attackers?"
query_result = voyage.embed([query], model="voyage-4", input_type="query")
query_vector = np.array(query_result.embeddings[0])

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

scores = [cosine_similarity(query_vector, v) for v in doc_vectors]
ranked = sorted(zip(names, scores), key=lambda x: -x[1])

print(f"Query: {query}\n")
for name, score in ranked:
    print(f"  {score:.4f}  {name}")

c:\AI-Training\Anthropic-learning\My-Anthropic-Training\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Query: Which vendor helps keep our systems safe from attackers?

  0.5353  BrightPath Security Solutions
  0.3220  Nimbus Cloud Services
  0.3179  Vertex Networking Group
  0.3108  Alderwood Office Supplies
  0.2135  Fixed-Term Consulting LLC


In [6]:
import re
from rank_bm25 import BM25Okapi

contracts = {
    "Nimbus Cloud Services": "Nimbus Cloud Services provides primary cloud infrastructure hosting including compute, storage, and managed database services for production workloads. Contract reference NCS-2025-0142. Annual value $63,000.",
    "BrightPath Security Solutions": "BrightPath Security Solutions provides endpoint security tooling, vulnerability scanning, and quarterly compliance reporting. Contract reference BPS-2025-0087. Annual value $24,000.",
    "Vertex Networking Group": "Vertex Networking Group provides managed WAN circuits and telecom support across all regional offices. Contract reference VNG-2025-0219. Annual value $19,500.",
    "Alderwood Office Supplies": "Alderwood Office Supplies supplies IT peripherals, cabling, and consumables for the office and data closet. Contract reference AOS-2025-0033. Annual value $4,200.",
    "Fixed-Term Consulting LLC": "Fixed-Term Consulting LLC provides contract engineering support for the infrastructure modernization project. Contract reference FTC-2025-0061. Annual value $47,000.",
}

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())  # strips punctuation so "NCS-2025-0142." still matches "NCS-2025-0142"

names = list(contracts.keys())
tokenized_docs = [tokenize(d) for d in contracts.values()]
bm25 = BM25Okapi(tokenized_docs)

def search(query):
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(zip(names, scores), key=lambda x: -x[1])
    print(f"Query: {query!r}")
    for name, score in ranked:
        print(f"  {score:6.3f}  {name}")
    print()

search("NCS-2025-0142")
search("Which vendor helps keep our systems safe from attackers?")

Query: 'NCS-2025-0142'
   2.237  Nimbus Cloud Services
   0.190  BrightPath Security Solutions
   0.190  Fixed-Term Consulting LLC
   0.187  Vertex Networking Group
   0.183  Alderwood Office Supplies

Query: 'Which vendor helps keep our systems safe from attackers?'
   0.000  Nimbus Cloud Services
   0.000  BrightPath Security Solutions
   0.000  Vertex Networking Group
   0.000  Alderwood Office Supplies
   0.000  Fixed-Term Consulting LLC



In [7]:
import re
import numpy as np
from rank_bm25 import BM25Okapi
from voyageai import Client
from anthropic import Anthropic

contracts = {
    "Nimbus Cloud Services": "Nimbus Cloud Services provides primary cloud infrastructure hosting including compute, storage, and managed database services for production workloads. Contract reference NCS-2025-0142. Auto-renews unless cancelled 30 days prior. Annual value $63,000.",
    "BrightPath Security Solutions": "BrightPath Security Solutions provides endpoint security tooling, vulnerability scanning, and quarterly compliance reporting. Contract reference BPS-2025-0087. Auto-renews unless cancelled 45 days prior. Annual value $24,000.",
    "Vertex Networking Group": "Vertex Networking Group provides managed WAN circuits and telecom support across all regional offices. Contract reference VNG-2025-0219. Auto-renews unless cancelled 60 days prior. Annual value $19,500.",
    "Alderwood Office Supplies": "Alderwood Office Supplies supplies IT peripherals, cabling, and consumables for the office and data closet. Contract reference AOS-2025-0033. Auto-renews unless cancelled 60 days prior. Annual value $4,200.",
    "Fixed-Term Consulting LLC": "Fixed-Term Consulting LLC provides contract engineering support for the infrastructure modernization project. Contract reference FTC-2025-0061. Fixed term, does not auto-renew. Annual value $47,000.",
}
names = list(contracts.keys())
docs = list(contracts.values())
query = "Which vendor helps keep our systems safe from attackers?"

# --- BM25 ranking ---
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())
bm25 = BM25Okapi([tokenize(d) for d in docs])
bm25_scores = bm25.get_scores(tokenize(query))
bm25_ranking = [n for n, _ in sorted(zip(names, bm25_scores), key=lambda x: -x[1])]

# --- Embedding ranking ---
voyage = Client()
doc_vecs = np.array(voyage.embed(docs, model="voyage-4", input_type="document").embeddings)
query_vec = np.array(voyage.embed([query], model="voyage-4", input_type="query").embeddings[0])
def cos_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
embed_scores = [cos_sim(query_vec, v) for v in doc_vecs]
embed_ranking = [n for n, _ in sorted(zip(names, embed_scores), key=lambda x: -x[1])]

# --- Reciprocal Rank Fusion ---
def reciprocal_rank_fusion(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            scores[doc] = scores.get(doc, 0) + 1 / (k + rank)
    return sorted(scores.items(), key=lambda x: -x[1])

fused = reciprocal_rank_fusion([bm25_ranking, embed_ranking])
print("Fused ranking:")
for doc, score in fused:
    print(f"  {score:.5f}  {doc}")

# --- Generation: pass the top result to Claude as context ---
top_contract_name = fused[0][0]
top_contract_text = contracts[top_contract_name]

client = Anthropic()
response = client.messages.create(
    model="claude-sonnet-5",
    max_tokens=300,
    messages=[{
        "role": "user",
        "content": (
            f"Context:\n{top_contract_name}: {top_contract_text}\n\n"
            f"Question: {query}\n\n"
            "Answer using only the context provided."
        ),
    }],
)
print(f"\nClaude's grounded answer:\n{next(b.text for b in response.content if b.type == 'text')}")

Fused ranking:
  0.03252  BrightPath Security Solutions
  0.03227  Nimbus Cloud Services
  0.03200  Vertex Networking Group
  0.03125  Alderwood Office Supplies
  0.03077  Fixed-Term Consulting LLC

Claude's grounded answer:
Based on the context provided, **BrightPath Security Solutions** helps keep systems safe from attackers.

They provide the following services related to security:
- **Endpoint security tooling**
- **Vulnerability scanning**
- **Quarterly compliance reporting**

Additional contract details:
- **Contract Reference:** BPS-2025-0087
- **Renewal Terms:** Auto-renews unless cancelled 45 days prior
- **Annual Value:** $24,000
